In [1]:
import os, glob
import numpy as np
import pandas as pd

for folder in ["raw", "bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)

!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip -O book-crossings.zip
!unzip -o -q book-crossings.zip -d raw
print(glob.glob("raw/**/*", recursive=True))

['raw/BX-Book-Ratings.csv', 'raw/BX-Books.csv', 'raw/BX-Users.csv']


In [2]:
def find(name):
    return glob.glob(f"raw/**/{name}", recursive=True)[0]

def read_raw(name):
    return pd.read_csv(find(name), sep=";", encoding="ISO-8859-1",
                       escapechar="\\", on_bad_lines="skip", dtype=str)

ts = pd.Timestamp.now()
files = {"BX-Users.csv": "users", "BX-Books.csv": "books", "BX-Book-Ratings.csv": "ratings"}

for fname, out in files.items():
    df = read_raw(fname)
    df["ingestion_timestamp"] = ts
    df.to_parquet(f"bronze/{out}.parquet", index=False)
    print(out, df.shape)

users (278858, 4)
books (271379, 9)
ratings (1149780, 4)


In [3]:
users   = pd.read_parquet("bronze/users.parquet")
books   = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")

for name, df in [("users", users), ("books", books), ("ratings", ratings)]:
    print("=====", name, df.shape)
    print(df.dtypes)
    print(df.isnull().sum())

===== users (278858, 4)
User-ID                        object
Location                       object
Age                            object
ingestion_timestamp    datetime64[us]
dtype: object
User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64
===== books (271379, 9)
ISBN                           object
Book-Title                     object
Book-Author                    object
Year-Of-Publication            object
Publisher                      object
Image-URL-S                    object
Image-URL-M                    object
Image-URL-L                    object
ingestion_timestamp    datetime64[us]
dtype: object
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            0
ingestion_timestamp    0
dtype: int64
===== ratings (1149780, 4)
User-ID                        o

**Table shapes, types and missing values**
- Users has 278,858 rows, books has 271,379 rows, and ratings has 1,149,780 rows.
- All columns were read as text (object) on purpose, so Bronze stays untouched. In Silver I will convert age, publication year and rating to numbers.
- Age has 110,762 missing values (about 40% of users). In Silver I will set impossible ages to null and fill all nulls with the median age.
- Books has 2 missing authors and 2 missing publishers. In Silver I will clean the text columns, and these rows will mostly drop out when I keep only books with enough ratings.
- Ratings has no missing values, so nothing to fix there.

In [4]:
r = pd.to_numeric(ratings["Book-Rating"], errors="coerce")
print(r.value_counts().sort_index())
print("Zero ratings %:", round((r == 0).mean() * 100, 1))

age = pd.to_numeric(users["Age"], errors="coerce")
print(age.describe())
print("Age < 5:", (age < 5).sum(), "| Age > 100:", (age > 100).sum())

year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
print(year.describe())
print("Year == 0:", (year == 0).sum(), "| Year > 2026:", (year > 2026).sum())

Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64
Zero ratings %: 62.3
count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64
Age < 5: 882 | Age > 100: 366
count    271379.000000
mean       1959.756050
std         258.011363
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Name: Year-Of-Publication, dtype: float64
Year == 0: 4619 | Year > 2026: 11


**Distribution of ratings, ages and publication years**
- Out of 1,149,780 ratings, 716,109 are 0, which is 62.3%. A 0 means "no explicit rating", not a bad rating, so in Silver I will remove all zeros and keep only ratings from 1 to 10.
- Among the real ratings, 8 is the most common (103,736), followed by 10 and 7, so users tend to rate generously.
- Age has impossible values: 882 users are below 5 and 366 are above 100, with a maximum of 244. In Silver I will set these to null and fill all nulls with the median age, which is about 32.
- Publication year has 4,619 books with year 0 and 11 books with a future year, with a maximum of 2050. In Silver I will set these to null.

In [5]:
explicit = ratings[r > 0]   # only real ratings, 1 to 10

per_user = explicit.groupby("User-ID").size()
per_book = explicit.groupby("ISBN").size()
print(per_user.describe()); print(per_book.describe())

for t in [1, 5, 10, 20]:
    print(f"users with >= {t} ratings:", (per_user >= t).sum(),
          "| books with >= {t} ratings:".replace("{t}", str(t)), (per_book >= t).sum())

sparsity = len(explicit) / (per_user.shape[0] * per_book.shape[0]) * 100
print(f"Filled cells: {sparsity:.4f}%  | Empty cells: {100 - sparsity:.4f}%")

count    77805.000000
mean         5.573819
std         44.001879
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       8524.000000
dtype: float64
count    185973.000000
mean          2.331903
std           6.834667
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         707.000000
dtype: float64
users with >= 1 ratings: 77805 | books with >= 1 ratings: 185973
users with >= 5 ratings: 14220 | books with >= 5 ratings: 14535
users with >= 10 ratings: 7334 | books with >= 10 ratings: 5645
users with >= 20 ratings: 3704 | books with >= 20 ratings: 2179
Filled cells: 0.0030%  | Empty cells: 99.9970%


**Ratings per user, ratings per book, and sparsity**
- After removing zeros, 77,805 users and 185,973 books have at least one real rating. Most have very few: the median user and the median book each have just 1 rating, while the busiest user has 8,524 and the busiest book has 707.
- Threshold choice: I will keep only users and books with at least 10 ratings. This keeps 7,334 users and 5,645 books. A threshold of 5 keeps more data but gives weaker similarity scores, and a threshold of 20 cuts the books down to 2,179, which leaves too few to recommend from. Ten is a balance between reliable similarity and enough data.
- The matrix is 99.997% empty, with only 0.003% of possible user-book pairs rated. In Gold I will store it as a sparse matrix.

In [6]:
from datetime import datetime
MIN_USER_RATINGS = 10
MIN_BOOK_RATINGS = 10

users   = pd.read_parquet("bronze/users.parquet").drop(columns="ingestion_timestamp")
books   = pd.read_parquet("bronze/books.parquet").drop(columns="ingestion_timestamp")
ratings = pd.read_parquet("bronze/ratings.parquet").drop(columns="ingestion_timestamp")

def clean_text(s):
    return s.str.strip().str.replace(r"\s+", " ", regex=True).str.title()

# ---------- USERS ----------
users = users.drop_duplicates()
users = users.rename(columns={"User-ID": "user_id", "Location": "location", "Age": "age"})
users["user_id"] = pd.to_numeric(users["user_id"], errors="coerce")
users = users.dropna(subset=["user_id"]).drop_duplicates(subset="user_id")
users["user_id"] = users["user_id"].astype(int)

users["age"] = pd.to_numeric(users["age"], errors="coerce")
users.loc[(users["age"] < 5) | (users["age"] > 100), "age"] = np.nan
users["age"] = users["age"].fillna(users["age"].median())

loc = users["location"].fillna("").str.split(",", n=2, expand=True)
for i in range(3):
    if i not in loc.columns:
        loc[i] = np.nan
users["city"], users["state"], users["country"] = loc[0], loc[1], loc[2]
for c in ["city", "state", "country"]:
    users[c] = clean_text(users[c]).replace({"": np.nan, "N/A": np.nan})
users = users.drop(columns="location")

# ---------- BOOKS ----------
books = books.drop_duplicates()
books = books.rename(columns={"ISBN": "isbn", "Book-Title": "title", "Book-Author": "author",
                              "Year-Of-Publication": "year", "Publisher": "publisher"})
books = books.drop(columns=[c for c in books.columns if c.startswith("Image-URL")])
books["isbn"] = books["isbn"].str.strip().str.upper()
books = books.drop_duplicates(subset="isbn")

books["year"] = pd.to_numeric(books["year"], errors="coerce")
books.loc[(books["year"] == 0) | (books["year"] > datetime.now().year), "year"] = np.nan
books["year"] = books["year"].astype("Int64")
for c in ["title", "author", "publisher"]:
    books[c] = clean_text(books[c])

# ---------- RATINGS ----------
ratings = ratings.drop_duplicates()
ratings = ratings.rename(columns={"User-ID": "user_id", "ISBN": "isbn", "Book-Rating": "rating"})
ratings["user_id"] = pd.to_numeric(ratings["user_id"], errors="coerce")
ratings["rating"]  = pd.to_numeric(ratings["rating"], errors="coerce")
ratings["isbn"]    = ratings["isbn"].str.strip().str.upper()
ratings = ratings.dropna(subset=["user_id", "rating"])
ratings["user_id"] = ratings["user_id"].astype(int)
ratings = ratings[(ratings["rating"] >= 1) & (ratings["rating"] <= 10)]
ratings = ratings.drop_duplicates(subset=["user_id", "isbn"], keep="last")

# keep only ratings whose user and book exist
ratings = ratings[ratings["user_id"].isin(users["user_id"]) & ratings["isbn"].isin(books["isbn"])]

# keep only active users and popular books (repeat until stable)
while True:
    uc = ratings["user_id"].value_counts()
    bc = ratings["isbn"].value_counts()
    new = ratings[ratings["user_id"].isin(uc[uc >= MIN_USER_RATINGS].index) &
                  ratings["isbn"].isin(bc[bc >= MIN_BOOK_RATINGS].index)]
    if len(new) == len(ratings):
        break
    ratings = new

users = users[users["user_id"].isin(ratings["user_id"])]
books = books[books["isbn"].isin(ratings["isbn"])]

users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)
print(users.shape, books.shape, ratings.shape)

(1820, 5) (2030, 5) (41469, 3)


**Silver layer: cleaning summary**
- Removed duplicate rows from users, books and ratings.
- Removed ratings of 0 and kept ratings from 1 to 10.
- Set ages below 5 or above 100 to null, then filled all nulls with the median age.
- Split Location into city, state and country.
- Set publication years of 0 or in the future to null.
- Stripped extra spaces, standardised case, and dropped the image-URL columns.
- Kept only ratings whose user and book exist in the users and books tables.
- Threshold: at least 10 ratings for both users and books, as justified in the EDA. Because removing weak users can push some books below 10, and the other way round, I repeated the filter until nothing changed.
- Final Silver sizes: 1,820 users, 2,030 books, 41,469 ratings.

In [7]:
from scipy.sparse import csr_matrix, save_npz

users   = pd.read_parquet("silver/users.parquet")
books   = pd.read_parquet("silver/books.parquet")
ratings = pd.read_parquet("silver/ratings.parquet")

user_codes, user_ids = pd.factorize(ratings["user_id"])
book_codes, isbns    = pd.factorize(ratings["isbn"])

matrix = csr_matrix((ratings["rating"].values.astype(float), (user_codes, book_codes)),
                    shape=(len(user_ids), len(isbns)))   # rows = users, columns = books
save_npz("gold/user_book_matrix.npz", matrix)

user_index = pd.DataFrame({"user_idx": range(len(user_ids)), "user_id": user_ids})
book_index = pd.DataFrame({"book_idx": range(len(isbns)), "isbn": isbns})
user_index.to_parquet("gold/user_index.parquet", index=False)

book_lookup = book_index.merge(books[["isbn", "title", "author"]], on="isbn", how="left")
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

user_agg = ratings.groupby("user_id")["rating"].agg(avg_rating="mean", num_ratings="count").reset_index()
book_agg = ratings.groupby("isbn")["rating"].agg(avg_rating="mean", num_ratings="count").reset_index()
user_agg.to_parquet("gold/user_aggregates.parquet", index=False)
book_agg.to_parquet("gold/book_aggregates.parquet", index=False)

print(matrix.shape, "non-empty cells:", matrix.nnz)

(1820, 2030) non-empty cells: 41469


**Gold layer: model-ready data**
- Built a sparse user-book rating matrix with 1,820 users as rows and 2,030 books as columns, and saved it as a sparse file. It has 41,469 filled cells, about 1.1% of all cells. That is far denser than the raw 0.003%, because of the minimum-10 filter.
- Saved index tables that map each matrix row to a user ID and each column to a book ID.
- Saved a book lookup table with ISBN, title and author, so recommendations show readable titles.
- Saved user-level and book-level aggregates, with the average rating and number of ratings for each.

In [8]:
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

ub = load_npz("gold/user_book_matrix.npz")            # users x books
user_index  = pd.read_parquet("gold/user_index.parquet")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")
book_agg    = pd.read_parquet("gold/book_aggregates.parquet")

book_lookup = book_lookup.merge(book_agg, on="isbn", how="left").sort_values("book_idx").reset_index(drop=True)
uid_to_idx = dict(zip(user_index["user_id"], user_index["user_idx"]))

bu = ub.T.tocsr()                                     # books x users
knn = NearestNeighbors(metric="cosine", algorithm="brute").fit(bu)
print(bu.shape)

(2030, 1820)


**Item-based collaborative filtering model**
- Loaded the sparse rating matrix from Gold and transposed it, so each row is a book and each column is a user (2,030 by 1,820).
- Fitted scikit-learn's NearestNeighbors with the cosine metric on this sparse matrix. Cosine distance is 1 minus cosine similarity, so the closest neighbours are the most similar books.


In [9]:
def find_book(title):
    m = book_lookup[book_lookup["title"].str.lower().str.contains(title.lower(), regex=False, na=False)]
    return None if m.empty else m.sort_values("num_ratings", ascending=False).iloc[0]

def similar_books(title, n=10):
    b = find_book(title)
    if b is None:
        return f"'{title}' not found in the data"
    dist, ind = knn.kneighbors(bu[int(b["book_idx"])], n_neighbors=n + 1)
    keep = ind[0] != b["book_idx"]
    idx, d = ind[0][keep][:n], dist[0][keep][:n]
    out = book_lookup.iloc[idx][["title", "author"]].copy()
    out["similarity"] = (1 - d).round(3)
    print("Books similar to:", b["title"])
    return out.reset_index(drop=True)

similar_books("Harry Potter")

Books similar to: Harry Potter And The Sorcerer'S Stone (Harry Potter (Paperback))


,title,author,similarity
0,Harry Potter And The Chamber Of Secrets (Book 2),J. K. Rowling,0.411
1,Harry Potter And The Prisoner Of Azkaban (Book 3),J. K. Rowling,0.316
2,Harry Potter And The Goblet Of Fire (Book 4),J. K. Rowling,0.254
3,Harry Potter And The Goblet Of Fire (Book 4),J. K. Rowling,0.254
4,Harry Potter And The Order Of The Phoenix (Boo...,J. K. Rowling,0.243
5,Harry Potter And The Prisoner Of Azkaban (Book 3),J. K. Rowling,0.204
6,Harry Potter And The Chamber Of Secrets (Book 2),J. K. Rowling,0.189
7,Fried Green Tomatoes At The Whistle Stop Cafe,Fannie Flagg,0.189
8,Love Story,Erich Segal,0.170
9,Anne Of Green Gables (Anne Of Green Gables Nov...,L.M. Montgomery,0.163


**Similar books function**
- Wrote similar_books(title), which finds the best-matching book by title and returns its 10 nearest neighbours with similarity scores.
- Sanity check with "Harry Potter": the top results are other Harry Potter books by J. K. Rowling, so the similarity makes sense.
- Some titles repeat because different editions of the same book have different ISBNs and are treated as separate items. Similarity drops quickly after the series books, because few users rated both.

In [10]:
def recommend(user_id, n=10, like_threshold=8, k=30):
    if user_id not in uid_to_idx:
        return "Unknown user"
    row = ub[uid_to_idx[user_id]]
    read = set(row.indices)
    mask = row.data >= like_threshold
    liked_idx, liked_rating = row.indices[mask], row.data[mask]
    if len(liked_idx) == 0:
        return "This user has no highly rated books"

    dist, ind = knn.kneighbors(bu[liked_idx], n_neighbors=k + 1)
    scores = {}
    for rating, d_row, i_row in zip(liked_rating, dist, ind):
        for d, j in zip(d_row, i_row):
            if j in read:
                continue
            scores[j] = scores.get(j, 0) + (1 - d) * rating

    top = sorted(scores.items(), key=lambda x: -x[1])[:n]
    out = book_lookup.iloc[[j for j, _ in top]][["title", "author"]].copy()
    out["score"] = [round(s, 2) for _, s in top]
    return out.reset_index(drop=True)

In [11]:
user_agg = pd.read_parquet("gold/user_aggregates.parquet")
demo_users = user_agg[user_agg["num_ratings"] >= 30].sample(3, random_state=42)["user_id"].tolist()

for uid in demo_users:
    row = ub[uid_to_idx[uid]]
    liked = row.indices[row.data >= 8]
    print("=" * 70, f"\nUser {uid} rated highly:")
    print(book_lookup.iloc[liked][["title", "author"]].head(10).to_string(index=False))
    print(f"\nRecommendations for user {uid}:")
    print(recommend(uid, n=10).to_string(index=False))

User 177432 rated highly:
                                                     title           author
                              Stephen King'S Danse Macabre     Stephen King
   C Is For Corpse (Kinsey Millhone Mysteries (Paperback))      Sue Grafton
                                         I Is For Innocent      Sue Grafton
                                                  Hideaway   Dean R. Koontz
                                              Dragon Tears   Dean R. Koontz
         The Drawing Of The Three (The Dark Tower, Book 2)     Stephen King
                                             The Dark Half     Stephen King
                                                      1984    George Orwell
The Fellowship Of The Ring (The Lord Of The Rings, Part 1) J. R. R. Tolkien
  The Clan Of The Cave Bear (Earth'S Children (Paperback))     Jean M. Auel

Recommendations for user 177432:
                                                            title         author  score
                

**Demo and sanity check**
- User 177432 rated highly Stephen King, Dean Koontz, Sue Grafton and Tolkien books. The recommendations are mostly Stephen King, such as Needful Things, Dolores Claiborne and the Dark Tower series, plus a Dean Koontz novel. This matches their taste in horror and thrillers.
- User 35836 rated highly Anne Rice, Jeffery Deaver and Stephen King books. The recommendations include another Lincoln Rhyme book by Jeffery Deaver, more Anne Rice vampire books, and several Stephen King novels. The model picked up all three of their interests.
- User 216683 rated highly several Nora Roberts books along with Jodi Picoult and John Irving. The recommendations are almost all Nora Roberts, which fits, because she was their most-represented author.
- Overall, the recommender suggests books by the same authors or in the same genre as the books each user liked, and it never recommends a book the user has already read. One limitation is that it can lean heavily on one author, as with the third user.